In [1]:
import json

notebook = {
    "cells": [
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "# **Notebook 01: Análise Exploratória e Tratamento do Dataset IPDM**\n",
                "\n",
                "### **1. Configuração do Ambiente e Importação de Pacotes**\n",
                "Carregamento das bibliotecas para manipulação de dados (`tidyverse`, `dplyr`, `tidyr`) e interpolação de strings (`glue`)."
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "library(glue)\n",
                "library(dplyr)\n",
                "library(tidyr)"
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "### **2. Carga dos Dados Brutos via GitHub**\n",
                "Construção dinâmica da URL do repositório remoto e importação do arquivo `dados_ipdm.csv`."
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "usuario <- 'Megalonnix'\n",
                "repositorio <- 'Projeto_Integrador_PI3_2Sem_2026'\n",
                "nm_dataset_usado <- 'dados_ipdm.csv'\n",
                "\n",
                "URL_GITHUB <- glue('https://raw.githubusercontent.com/{usuario}/{repositorio}/main/{nm_dataset_usado}')\n",
                "\n",
                "df <- read.csv(URL_GITHUB, sep = \";\", dec = \",\", fileEncoding = 'Latin1')\n",
                "\n",
                "# Visualização inicial do dataframe bruto\n",
                "head(df)\n",
                "glimpse(df)"
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "### **3. Inspeção Inicial e Padronização do Schema**\n",
                "Remoção da coluna residual `X` e renomeação das colunas para o padrão *snake_case*."
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "df <- df %>% select(-X)\n",
                "\n",
                "colnames(df) <- c(\n",
                "  \"cod_ibge\", \n",
                "  \"municipio\", \n",
                "  \"valor\", \n",
                "  \"ano\", \n",
                "  \"tipo\", \n",
                "  \"valor_estado\", \n",
                "  \"indicador_1\", \n",
                "  \"indicador_2\", \n",
                "  \"indicador_3\", \n",
                "  \"indicador_4\", \n",
                "  \"indicador_5\"\n",
                ")\n",
                "\n",
                "# Inspeção do schema renomeado\n",
                "head(df, 10)\n",
                "glimpse(df)"
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "### **4. Sanitização e Limpeza dos Dados**\n",
                "Conversão de vírgula para ponto, tratamento do tipo numérico e descarte de registros nulos ou zerados."
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "df$valor <- as.numeric(gsub(\",\", \".\", df$valor))\n",
                "df$valor_estado <- as.numeric(gsub(\",\", \".\", df$valor_estado))\n",
                "\n",
                "df <- df[!is.na(df$valor) & df$valor != 0, ]\n",
                "df <- df[!is.na(df$municipio) & trimws(df$municipio) != \"\", ]\n",
                "df <- df[!is.na(df$tipo) & trimws(df$tipo) != \"\", ]\n",
                "\n",
                "# Visualização do dataframe limpo e resumo estatístico\n",
                "head(df, 10)\n",
                "summary(df)"
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "### **5. Diagnóstico de Domínios e Segregação de Escalas**\n",
                "Separação entre Notas Sintéticas (0 a 1) e Dados Brutos com rotinas de verificação dos limites numéricos."
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "df_notas  <- df %>% filter(trimws(indicador_5) != \"\" | (valor <= 1 & valor_estado <= 1))\n",
                "df_brutos <- df %>% filter(trimws(indicador_5) == \"\" & (valor > 1 | valor_estado > 1))\n",
                "\n",
                "# Visualização prévia de cada subset gerado\n",
                "head(df_notas, 5)\n",
                "head(df_brutos, 5)\n",
                "\n",
                "get_resumo_valores <- function(df_notas, df_brutos) {\n",
                "  data.frame(\n",
                "    Dataframe = c(\"Notas (0 a 1)\", \"Notas (0 a 1)\", \"Dados Brutos\", \"Dados Brutos\"),\n",
                "    Coluna    = c(\"valor\", \"valor_estado\", \"valor\", \"valor_estado\"),\n",
                "    Minimo    = c(min(df_notas$valor, na.rm=TRUE), min(df_notas$valor_estado, na.rm=TRUE), \n",
                "                  min(df_brutos$valor, na.rm=TRUE), min(df_brutos$valor_estado, na.rm=TRUE)),\n",
                "    Maximo    = c(max(df_notas$valor, na.rm=TRUE), max(df_notas$valor_estado, na.rm=TRUE), \n",
                "                  max(df_brutos$valor, na.rm=TRUE), max(df_brutos$valor_estado, na.rm=TRUE))\n",
                "  )\n",
                "}\n",
                "\n",
                "# Exibição da comparação dos limites numéricos\n",
                "get_resumo_valores(df_notas, df_brutos)"
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "### **6. Despivoteamento (`Pivot`) e Unificação das Métricas Brutas**\n",
                "Reestruturação de formato amplo (*wide*) para longo (*long*) unificando indicadores em `metrica`."
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "df_ipdm <- df %>%\n",
                "  filter(indicador_5 != \"\") %>%\n",
                "  select(-indicador_1, -indicador_2, -indicador_3, -indicador_4)\n",
                "\n",
                "# Visualização do dataframe do IPDM\n",
                "head(df_ipdm, 10)\n",
                "\n",
                "df_brutos_unificados <- df %>%\n",
                "  filter(indicador_5 == \"\") %>%\n",
                "  pivot_longer(\n",
                "    cols = indicador_1:indicador_4,\n",
                "    names_to = \"coluna_origem\",\n",
                "    values_to = \"metrica\"\n",
                "  ) %>%\n",
                "  filter(metrica != \"\") %>%\n",
                "  select(-coluna_origem, -indicador_5)\n",
                "\n",
                "# Visualização do resultado unificado do pivot\n",
                "head(df_brutos_unificados, 10)\n",
                "\n",
                "lista_variaveis <- split(df_brutos_unificados, df_brutos_unificados$metrica)\n",
                "\n",
                "# Lista de nomes das métricas separadas\n",
                "names(lista_variaveis)"
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "### **7. Agrupamento e Extração Modular por Dimensões Socioeconômicas**\n",
                "Separação das variáveis em dataframes individuais por dimensão (Longevidade, Escolaridade e Riqueza)."
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "# --- DIMENSÃO: LONGEVIDADE ---\n",
                "df_mort_infantil  <- lista_variaveis[[\"Taxas de mortalidade infantil (por mil nascidos vivos)\"]]\n",
                "df_mort_perinatal <- lista_variaveis[[\"Taxas de mortalidade perinatal (por mil nascidos vivos)\"]]\n",
                "df_mort_15_39     <- lista_variaveis[[\"Taxas de mortalidade 15 a 39 anos (por mil hab.)\"]]\n",
                "df_mort_60_69     <- lista_variaveis[[\"Taxas de mortalidade  60 a 69 anos (por mil hab.)\"]]\n",
                "\n",
                "# --- DIMENSÃO: ESCOLARIDADE ---\n",
                "df_atendimento_0_3 <- lista_variaveis[[\"Taxas de atendimento escolar de crianças de 0 a 3 anos (%)\"]]\n",
                "df_prof_5ano        <- lista_variaveis[[\"Proporção média de alunos do 5º ano com proficiência em Língua Portuguesa e Matemática (%)\"]]\n",
                "df_prof_9ano        <- lista_variaveis[[\"Proporção média de  alunos do 9º ano com proficiência em Língua Portuguesa e Matemática (%)\"]]\n",
                "df_distorcao_em     <- lista_variaveis[[\"Taxas de distorção idade-série no Ensino Médio (%)\"]]\n",
                "\n",
                "# --- DIMENSÃO: RIQUEZA ---\n",
                "df_pib         <- lista_variaveis[[\"Produto Interno Bruto per capita (R$ de 2024)\"]]\n",
                "df_renda       <- lista_variaveis[[\"Rendimento do trabalho formal mais benefícios previdenciarios per capita (R$ de 2024)\"]]\n",
                "df_energia_res <- lista_variaveis[[\"Consumo anual de energia elétrica residencial (MWh) por ligação\"]]\n",
                "df_energia_com <- lista_variaveis[[\"Consumo anual de energia elétrica comercial, serviços e rural (MWh) por ligação\"]]\n",
                "\n",
                "metricas_desenvolvimento <- list(\n",
                "  longevidade_1 = df_mort_infantil, \n",
                "  longevidade_2 = df_mort_perinatal,\n",
                "  longevidade_3 = df_mort_15_39, \n",
                "  longevidade_4 = df_mort_60_69, \n",
                "  escolaridade_1 = df_atendimento_0_3, \n",
                "  escolaridade_2 = df_prof_5ano, \n",
                "  escolaridade_3 = df_prof_9ano, \n",
                "  escolaridade_4 = df_distorcao_em, \n",
                "  economia_1     = df_pib, \n",
                "  economia_2     = df_renda, \n",
                "  consumo_1      = df_energia_res, \n",
                "  consumo_2      = df_energia_com\n",
                ")\n",
                "\n",
                "# Exibição das amostras finais de alguns dataframes isolados\n",
                "head(df_mort_infantil, 5)\n",
                "head(df_pib, 5)\n",
                "head(df_atendimento_0_3, 5)"
            ]
        }
    ],
    "metadata": {
        "kernelspec": {
            "display_name": "R",
            "language": "R",
            "name": "ir"
        },
        "language_info": {
            "file_extension": ".r",
            "mimetype": "text/x-r-source",
            "name": "R",
            "pygments_lexer": "r",
            "version": "4.2.0"
        }
    },
    "nbformat": 4,
    "nbformat_minor": 2
}

# Salva o arquivo .ipynb no diretório atual
with open("notebook_ipdm.ipynb", "w", encoding="utf-8") as f:
    json.dump(notebook, f, ensure_ascii=False, indent=2)

print("Arquivo 'notebook_ipdm.ipynb' gerado com sucesso!")

Arquivo 'notebook_ipdm.ipynb' gerado com sucesso!
